<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/population_est_ssd_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map

In [ ]:
roi = ee.Geometry.Point(map.draw_last_feature.geometry().coordinates().getInfo())
roi

In [ ]:
border = (ee.FeatureCollection("FAO/GAUL/2015/level0")

  .filterBounds(roi)
  .geometry()
  .simplify(100)
)
vis_params = {"color":"yellow"}
map.addLayer(border, vis_params, "border")

In [ ]:
pop = ee.ImageCollection("JRC/GHSL/P2023A/GHS_POP")
pop

In [ ]:
pop_stack = pop.toBands().clip(roi)
pop_stack

In [ ]:
import geemap

map = geemap.Map()
map.add("basemap_selector")
map.addLayer(pop_stack,{},'pop_stack')

In [ ]:
def pop_count(img):
  pop_sum = img.reduceRegion(reducer = ee.Reducer.sum(), geometry = border, scale = 100, bestEffort=True, maxPixels=1e9).values().get(0)
  date = img.date().format('YYYY-MM-dd')
  return ee.Feature(None, {'date': date, 'pop': pop_sum})

In [ ]:
pop_val = pop.map(pop_count)
feature_list = pop_val.aggregate_array('pop').getInfo()
date_list = pop_val.aggregate_array('date').getInfo()
pop_val

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
pop_sum = feature_list
pop_sum

In [ ]:
import matplotlib.pyplot as plt

# Create a DataFrame
df = pd.DataFrame({'Date': date_list, 'Population': pop_sum})

# Convert 'Date' column to datetime objects
df['Date'] = pd.to_datetime(df['Date'])

# Plot the population over time
plt.figure(figsize=(10, 6))
plt.plot(df['Date'], df['Population'], marker='o')
plt.title('Population Over Time')
plt.xlabel('Date')
plt.ylabel('Population')
plt.grid(True)
plt.show()

In [ ]:
# Set 'Date' as the index for calculations
df_index = df.set_index('Date')

# Calculate the population growth rate (percentage change)
df_index['growth_rate'] = df_index['Population'].pct_change() * 100

# Smooth the growth rate using a rolling mean (e.g., a 2-period window)
growth_rate_smoothed = df_index['growth_rate'].rolling(window=2, min_periods=1).mean()

# Display the DataFrame with growth rates
display(df_index)

In [ ]:
# Plotting Growth Rate (Original vs. Smoothed)

plt.figure(figsize=(10, 6))
df_index['growth_rate'].plot(label='Original Growth Rate', legend=True)
growth_rate_smoothed.plot(label='Smoothed Growth Rate', legend=True)
plt.title('South Sudan Population Change 1980-2030')
plt.xlabel('Date')
plt.ylabel('South Sudan Population Change 1980-2030')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
df_index['growth_rate'].plot(label='Population Growth Rate', legend=True)
plt.title('Original vs. Smoothed Annual Population')
plt.xlabel('Date')
plt.ylabel('Growth Rate (%)')
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(df['Date'], df['Population'] / 1_000_000, marker='o')
plt.title('South Sudan Population Over Time (in Millions)')
plt.xlabel('Date')
plt.ylabel('Population (Millions)')
plt.grid(True)
plt.show()

In [ ]:
plt.savefig("ssd_population.png", dpi = 365, bbox_inches = 'tight')

In [ ]:
plt.figure(figsize=(10, 6))
df_index['growth_rate'].plot(label='Population Change', legend=True)
growth_rate_smoothed.plot(label='Smoothed Population Change', legend=True)
plt.title('Population Growth Change 1980-2030')
plt.xlabel('Date')
plt.ylabel('Population Change')
plt.grid(True)
plt.show()